# Baseline Training

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib
base_dir = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd().parents[1]
models_dir = base_dir / 'models'
results_dir = base_dir / 'results'
data_dir = base_dir / 'data'
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)
NUMERIC = ['amount','transaction_time','device_score']
CATEGORICAL = ['merchant_type','card_country']


Generate synthetic dataset

In [ ]:
rng = np.random.default_rng(42)
n = 5000
amount = rng.gamma(2.0, 50.0, n)
transaction_time = rng.integers(0, 24*3600, n)
merchant_type = rng.choice(['grocery','online','travel','foreign_high_risk'], n, p=[0.4,0.3,0.2,0.1])
card_country = rng.choice(['US','UK','IN','CN','BR'], n)
device_score = rng.random(n)
risk = ((amount>200).astype(int) + (merchant_type=='foreign_high_risk').astype(int) + (device_score>0.7).astype(int))
y = (risk + rng.integers(0,2,n)) >= 2
df = pd.DataFrame({'amount':amount, 'transaction_time':transaction_time.astype(float), 'merchant_type':merchant_type, 'card_country':card_country, 'device_score':device_score, 'label':y.astype(int)})
df.head()


Handle imbalance via oversampling

In [ ]:
maj = df[df.label==0]
mino = df[df.label==1]
factor = max(1, len(maj)//max(1,len(mino)))
mino_os = pd.concat([mino]*factor, ignore_index=True).iloc[:len(maj)]
df_os = pd.concat([maj, mino_os], ignore_index=True).sample(frac=1.0, random_state=42)
X = df_os[NUMERIC + CATEGORICAL]
y = df_os['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


Preprocessing pipeline and models

In [ ]:
ct = ColumnTransformer([('num', StandardScaler(), NUMERIC), ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)])
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
pipe_lr = Pipeline([('ct', ct), ('clf', lr)])
pipe_rf = Pipeline([('ct', ct), ('clf', rf)])
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)


Metrics

In [ ]:
proba_lr = pipe_lr.predict_proba(X_test)[:,1]
proba_rf = pipe_rf.predict_proba(X_test)[:,1]
preds_rf = (proba_rf>=0.5).astype(int)
metrics = {
 'accuracy': float(accuracy_score(y_test, preds_rf)),
 'precision': float(precision_score(y_test, preds_rf)),
 'recall': float(recall_score(y_test, preds_rf)),
 'f1': float(f1_score(y_test, preds_rf)),
 'roc_auc': float(roc_auc_score(y_test, proba_rf)),
 'confusion_matrix': confusion_matrix(y_test, preds_rf).tolist(),
 'baseline_auc': float(roc_auc_score(y_test, proba_lr)),
}
metrics


Save artifacts and sample transactions

In [ ]:
joblib.dump(pipe_rf.named_steps['clf'], models_dir / 'fraud_model.pkl')
joblib.dump(ct, models_dir / 'preprocessing_pipeline.pkl')
(results_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
df.sample(12, random_state=42).drop(columns=['label']).to_csv(data_dir / 'sample_transactions.csv', index=False)
